# TrustGuard — prediction notebook (Track 2)
**This is the file uploaded to the portal (card 1), next to `model.pkl` (card 2).**
It follows the organisers' `template_notebook.ipynb`: the sandbox runs it top-to-bottom with no internet and
no `train.csv`; the test file (`DATATHON_INPUT_PATH`) has **no `id` column and shuffled rows**; we write one
probability per row, same order, under a single `prediction` column to `DATATHON_OUTPUT_PATH`.
One upload per 2h and a failed run burns the window — always dry-run locally first.

The feature code below is inlined from `notebooks/submission.ipynb` §3–4 and must stay byte-identical to it.
`model.pkl` must be trained under the platform's versions (`.venv-image`, from `requirements-image.txt`).


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")


In [ ]:
# --- feature engineering: keep byte-identical to the training notebook (§3-4) ---
MERCHANTS = ["cash_transfer", "electronics", "entertainment", "food_delivery", "gaming",
             "grocery", "luxury", "retail", "transport", "travel", "utilities"]
COUNTRIES = ["AU", "GB", "ID", "JP", "MY", "PH", "SG", "TH", "US", "VN"]
CHANNELS  = ["bank_transfer", "card_present", "ecommerce", "mobile_app"]
CAT_VOCAB = {"merchant_category": MERCHANTS, "country": COUNTRIES, "transaction_channel": CHANNELS}
IMPUTE = {"transactions_last_24h": 4.0, "spend_last_24h": 217.515, "account_age": 790.0,
          "new_device": 0.0, "transactions_last_1h": 1.0,
          "merchant_category": "grocery", "country": "SG", "transaction_channel": "card_present"}
NUMS = ["log_amount", "log_age", "log_spend", "transactions_last_1h", "transactions_last_24h",
        "transaction_hour", "amt_share_24h", "amt_vs_prev_avg", "log_amt_per_age", "burst_ratio",
        "log_amt_x_newdev", "log_amt_x_night", "hour_sin", "hour_cos", "spend_per_txn"]
FLAGS = ["new_device", "night", "amt_gt500", "age_lt180", "n1_ge3", "n1_ge4", "n24_ge10",
         "newdev_young", "big_old", "night_newdev", "burst_young"]
FEATURES = list(CAT_VOCAB) + NUMS + FLAGS

def make_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    for c, v in IMPUTE.items():
        d[c] = (pd.to_numeric(d[c], errors="coerce") if c not in CAT_VOCAB else d[c]).fillna(v)
    for c in CAT_VOCAB:
        d[c] = d[c].where(d[c].isin(CAT_VOCAB[c]), "unknown")
    a, s, age = d.transaction_amount.astype(float), d.spend_last_24h, d.account_age
    n1, n24, nd, hr = d.transactions_last_1h, d.transactions_last_24h, d.new_device, d.transaction_hour
    X = d[list(CAT_VOCAB)].copy()
    X["log_amount"] = np.log1p(a)
    X["log_age"] = np.log1p(age)
    X["log_spend"] = np.log1p(s)
    X["transactions_last_1h"] = n1
    X["transactions_last_24h"] = n24
    X["transaction_hour"] = hr
    X["amt_share_24h"] = a / (a + s)
    X["amt_vs_prev_avg"] = np.log1p(a) - np.log1p(s / np.maximum(n24 - 1, 1))
    X["log_amt_per_age"] = np.log1p(a) - np.log1p(age)
    X["burst_ratio"] = n1 / np.maximum(n24, 1)
    X["new_device"] = nd
    X["night"] = (hr <= 5).astype(float)
    X["amt_gt500"] = (a > 500).astype(float)
    X["age_lt180"] = (age < 180).astype(float)
    X["n1_ge3"] = (n1 >= 3).astype(float)
    X["n1_ge4"] = (n1 >= 4).astype(float)
    X["n24_ge10"] = (n24 >= 10).astype(float)
    X["newdev_young"] = nd * (age < 180)
    X["big_old"] = ((a > 500) & (age > 1000)).astype(float)
    X["log_amt_x_newdev"] = X["log_amount"] * nd
    X["log_amt_x_night"] = X["log_amount"] * X["night"]
    X["hour_sin"] = np.sin(2 * np.pi * hr / 24)
    X["hour_cos"] = np.cos(2 * np.pi * hr / 24)
    X["spend_per_txn"] = np.log1p(s / np.maximum(n24, 1))
    X["night_newdev"] = X["night"] * nd
    X["burst_young"] = X["n1_ge3"] * (age < 365).astype(float)
    return X[FEATURES]


In [ ]:
model, features = joblib.load("model.pkl")
test_df = pd.read_csv(TEST_PATH)  # sandbox: no id column, shuffled rows — features never use id
X_test = make_features(test_df)[features]
if hasattr(model, "predict_proba"):
    preds = model.predict_proba(X_test)[:, 1]
else:  # lightgbm Booster: predict() already returns P(fraud)
    preds = model.predict(X_test)

assert len(preds) == len(test_df)
assert 0 <= preds.min() <= preds.max() <= 1
pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}, mean p {preds.mean():.4f}")
